# Prompt 19: Reading and Writing DataFrames
## Databricks Certified Associate Developer for Apache Spark
### Topic 3 — DataFrame and Column Operations (30%)

---

## Read API — Two Equivalent Forms

```python
# Form 1: generic format-based builder (most flexible)
df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/path')

# Form 2: format shorthand (convenience methods)
df = spark.read.csv('/path', header=True, inferSchema=True)
df = spark.read.json('/path')
df = spark.read.parquet('/path')
df = spark.read.orc('/path')
```

Both forms produce the same result. Use `.format()` when you need many options or a dynamic format string.

## Write Modes — What Each Does

| Mode | Behaviour | Default? |
|------|-----------|----------|
| `'overwrite'` | Delete existing data at path, write new | No |
| `'append'` | Add new files alongside existing files | No |
| `'ignore'` | Do nothing if path/table exists (silent no-op) | No |
| `'error'` | Raise `AnalysisException` if path/table exists | **Yes** |
| `'errorIfExists'` | Alias for `'error'` | **Yes** |

```python
df.write.mode('overwrite').parquet('/output/path')
df.write.mode('append').csv('/output/path')
df.write.mode('ignore').json('/output/path')
```

## CSV Key Options

| Option | Type | Default | Description |
|--------|------|---------|-------------|
| `header` | bool | `False` | First row is a column header |
| `inferSchema` | bool | `False` | Infer column types (triggers an extra scan) |
| `sep` / `delimiter` | str | `,` | Field delimiter character |
| `nullValue` | str | `''` | String to treat as NULL |
| `nanValue` | str | `'NaN'` | String to treat as NaN |
| `dateFormat` | str | `'yyyy-MM-dd'` | Format for parsing dates |
| `timestampFormat` | str | ISO 8601 | Format for parsing timestamps |
| `encoding` | str | `'UTF-8'` | File encoding |
| `multiLine` | bool | `False` | Allow field values spanning multiple lines |
| `quote` | str | `'"'` | Quote character |
| `escape` | str | `'\\'` | Escape character |

## JSON Key Options

| Option | Default | Description |
|--------|---------|-------------|
| `multiLine` | `False` | Each file is a single JSON object (not one-JSON-per-line) |
| `allowSingleQuotes` | `False` | Accept single-quoted strings |
| `dateFormat` | `'yyyy-MM-dd'` | Date format string |

## partitionBy — Write vs Read

```python
# Writing: creates a directory hierarchy  /output/year=2024/month=01/part-00000.parquet
df.write.mode('overwrite').partitionBy('year', 'month').parquet('/output')

# Reading: Spark automatically does partition pruning when filter matches partition columns
df2 = spark.read.parquet('/output')
df2.filter(col('year') == 2024).filter(col('month') == 1)  # reads only year=2024/month=01/
```

In [ ]:
# Cell 1: Setup — write sample DataFrames to temp files for reading exercises
import os
import tempfile
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType, BooleanType
)

spark = SparkSession.builder \
    .appName('ReadingWriting') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Base temp directory
BASE = tempfile.mkdtemp().replace('\\', '/')
print(f'Temp directory: {BASE}')

# Sample sales DataFrame
sales = spark.createDataFrame([
    (1, 'Alice',   'Electronics', 1200.0, '2024-01-15', 'US'),
    (2, 'Bob',     'Clothing',     450.0, '2024-01-22', 'UK'),
    (3, 'Carol',   'Electronics',  800.0, '2024-02-10', 'US'),
    (4, 'Dave',    'Books',         95.0, '2024-02-28', 'CA'),
    (5, 'Eve',     'Electronics', 1500.0, '2024-03-05', 'US'),
    (6, 'Frank',   'Clothing',     320.0, '2024-03-18', 'UK'),
    (7, None,      'Books',        None,  None,         'US'),  # row with nulls
], ['sale_id', 'customer', 'category', 'amount', 'sale_date', 'region'])

print('=== Sample data ===')
sales.show()
sales.printSchema()

# Write CSV with header for read exercises
csv_path = f'{BASE}/sales_csv'
sales.write.mode('overwrite').option('header', True).csv(csv_path)
print(f'Written CSV to: {csv_path}')

# Write Parquet for read exercises
parquet_path = f'{BASE}/sales_parquet'
sales.write.mode('overwrite').parquet(parquet_path)
print(f'Written Parquet to: {parquet_path}')

# Write JSON for read exercises
json_path = f'{BASE}/sales_json'
sales.write.mode('overwrite').json(json_path)
print(f'Written JSON to: {json_path}')

In [ ]:
# Cell 2: Reading CSV

print('=== Read CSV without options — everything is StringType ===')
df_no_opts = spark.read.csv(csv_path)
df_no_opts.printSchema()   # _c0, _c1 ... all StringType, header becomes a data row
df_no_opts.show(3)

print('=== Read CSV with header=True, inferSchema=True ===')
df_infer = spark.read.csv(csv_path, header=True, inferSchema=True)
df_infer.printSchema()     # column names from header, types inferred
df_infer.show()

print('=== Read CSV with explicit schema (preferred — no extra scan) ===')
schema = StructType([
    StructField('sale_id',   IntegerType(), True),
    StructField('customer',  StringType(),  True),
    StructField('category',  StringType(),  True),
    StructField('amount',    DoubleType(),  True),
    StructField('sale_date', StringType(),  True),
    StructField('region',    StringType(),  True),
])
df_explicit = spark.read.schema(schema).option('header', True).csv(csv_path)
df_explicit.printSchema()
df_explicit.show()

print('=== CSV option: sep (tab-separated) ===')
tsv_path = f'{BASE}/sales_tsv'
sales.write.mode('overwrite').option('header', True).option('sep', '\t').csv(tsv_path)
df_tsv = spark.read.option('header', True).option('sep', '\t').option('inferSchema', True).csv(tsv_path)
df_tsv.show()

print('=== CSV option: nullValue ===')
# Treat the string 'N/A' as NULL on read
df_null = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .option('nullValue', 'N/A') \
    .csv(csv_path)
df_null.show()

print('=== Format-based builder equivalent ===')
df_fmt = spark.read \
    .format('csv') \
    .option('header', True) \
    .option('inferSchema', True) \
    .load(csv_path)
print(f'format builder row count: {df_fmt.count()}')
assert df_fmt.count() == df_infer.count()  # identical result

In [ ]:
# Cell 3: Reading Parquet and JSON

print('=== Read Parquet — schema embedded in file, no options needed ===')
df_parquet = spark.read.parquet(parquet_path)
df_parquet.printSchema()   # types preserved from write — no inferSchema needed
df_parquet.show()

# Format builder
df_parquet2 = spark.read.format('parquet').load(parquet_path)
assert df_parquet.count() == df_parquet2.count()

print('=== Read JSON — one JSON object per line (default) ===')
df_json = spark.read.json(json_path)
df_json.printSchema()   # JSON schema inferred automatically
df_json.show()

print('=== Read JSON: multiLine=True — entire file is one JSON object ===')
# Create a multiLine JSON file for demonstration
import json
multi_json_path = f'{BASE}/multi.json'
sample = [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]
# Write as a proper JSON array (multiLine format)
with open(multi_json_path, 'w') as f:
    json.dump(sample, f, indent=2)

df_multi = spark.read.option('multiLine', True).json(multi_json_path)
df_multi.show()

print('=== Reading multiple files with glob pattern ===')
# Read all CSV files matching a pattern
df_glob = spark.read.option('header', True).option('inferSchema', True) \
               .csv(f'{csv_path}/*.csv')
print(f'Rows from glob: {df_glob.count()}')

# Read from a list of paths
import os
csv_files = [f'{csv_path}/{f}' for f in os.listdir(csv_path) if f.endswith('.csv')]
if csv_files:
    df_list = spark.read.option('header', True).option('inferSchema', True).csv(csv_files)
    print(f'Rows from list: {df_list.count()}')

In [ ]:
# Cell 4: Write modes

out = f'{BASE}/write_modes'

print('=== mode(overwrite) — replaces existing data ===')
sales.write.mode('overwrite').parquet(f'{out}/overwrite')
sales.write.mode('overwrite').parquet(f'{out}/overwrite')  # second write succeeds (replaces)
print(f'overwrite count: {spark.read.parquet(f"{out}/overwrite").count()}')  # 7

print('=== mode(append) — adds to existing data ===')
sales.write.mode('overwrite').parquet(f'{out}/append_test')
sales.write.mode('append').parquet(f'{out}/append_test')   # appends another 7 rows
print(f'append count: {spark.read.parquet(f"{out}/append_test").count()}')  # 14

print('=== mode(ignore) — silent no-op if path exists ===')
sales.write.mode('overwrite').parquet(f'{out}/ignore_test')
# Writing different data — but mode=ignore so nothing changes
sales.limit(3).write.mode('ignore').parquet(f'{out}/ignore_test')
print(f'ignore count: {spark.read.parquet(f"{out}/ignore_test").count()}')  # still 7

print('=== mode(error) / mode(errorIfExists) — raises exception if path exists ===')
sales.write.mode('overwrite').parquet(f'{out}/error_test')
try:
    sales.write.mode('error').parquet(f'{out}/error_test')  # path exists → exception
except Exception as e:
    print(f'Expected error: {type(e).__name__}')

# Default mode — same as 'error'
import shutil
if os.path.exists(f'{out}/default_test'):
    shutil.rmtree(f'{out}/default_test')
sales.write.parquet(f'{out}/default_test')   # no .mode() → default is 'error'
print(f'default mode write succeeded: {spark.read.parquet(f"{out}/default_test").count()}')

In [ ]:
# Cell 5: partitionBy and partition pruning

# Add year/month columns for partitioning
sales_with_date = sales.withColumn(
    'sale_date_parsed', F.to_date(col('sale_date'), 'yyyy-MM-dd')
).withColumn(
    'year',  F.year(col('sale_date_parsed'))
).withColumn(
    'month', F.month(col('sale_date_parsed'))
).drop('sale_date_parsed')

sales_with_date.show()

partitioned_path = f'{BASE}/sales_partitioned'

print('=== Write partitioned by year and month ===')
sales_with_date.write \
    .mode('overwrite') \
    .partitionBy('year', 'month') \
    .parquet(partitioned_path)

# Show directory structure created by partitionBy
print('Directory structure:')
for root, dirs, files in os.walk(partitioned_path):
    level = root.replace(partitioned_path, '').count(os.sep)
    indent = ' ' * 2 * level
    if files:
        print(f'{indent}{os.path.basename(root)}/')
        for f in files:
            print(f'{indent}  {f}')

print('=== Read all partitions ===')
df_all = spark.read.parquet(partitioned_path)
df_all.show()
print(f'Total rows: {df_all.count()}')

print('=== Partition pruning — only reads year=2024/month=1/ directory ===')
df_jan = spark.read.parquet(partitioned_path) \
              .filter(col('year') == 2024) \
              .filter(col('month') == 1)
df_jan.show()
print(f'Jan rows: {df_jan.count()}')

# Explain shows PartitionFilters in physical plan
df_jan.explain()

print('=== coalesce(1) before write — produces a single output file ===')
single_file_path = f'{BASE}/single_file'
sales_with_date.coalesce(1).write.mode('overwrite').parquet(single_file_path)
output_files = [f for f in os.listdir(single_file_path) if f.endswith('.parquet')]
print(f'Number of parquet files written: {len(output_files)}')

In [ ]:
# Cell 6: Delta format and saveAsTable

delta_path = f'{BASE}/sales_delta'

print('=== Write Delta format ===')
sales.write.mode('overwrite').format('delta').save(delta_path)
print(f'Delta written to: {delta_path}')

print('=== Read Delta format ===')
df_delta = spark.read.format('delta').load(delta_path)
df_delta.show()
df_delta.printSchema()

print('=== saveAsTable — writes and registers as a managed table ===')
# This creates a Hive-metastore-backed table accessible via spark.table() and spark.sql()
sales.write.mode('overwrite').saveAsTable('sales_managed')

# Read back via spark.table()
df_tbl = spark.table('sales_managed')
df_tbl.show()
print(f'spark.table() row count: {df_tbl.count()}')

# Also accessible via spark.sql()
spark.sql('SELECT category, COUNT(*) as cnt FROM sales_managed GROUP BY category ORDER BY cnt DESC').show()

print('=== Read Delta as table (when registered in metastore) ===')
sales.write.mode('overwrite').format('delta').saveAsTable('sales_delta_table')
spark.table('sales_delta_table').show()

print('=== Inspect schema with printSchema and dtypes ===')
df_delta.printSchema()
print(df_delta.dtypes)   # list of (column_name, type_string) tuples

spark.stop()
print('Done.')

## inferSchema vs Explicit Schema — Trade-offs

| | `inferSchema=True` | Explicit `StructType` schema |
|--|--------------------|---------------------------------|
| Extra scan | **Yes** — reads data twice | **No** — schema applied on first read |
| Correctness | May guess wrong type (int as long, date as string) | Exact — you control types |
| Null handling | Silently converts bad values to `null` | Silently converts bad values to `null` |
| Production use | OK for exploration / small files | **Preferred** for production pipelines |
| Schema drift | Fails silently if new columns appear | `allowMissingColumns` controls it |

```python
# Explicit schema example
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField('id',       IntegerType(), nullable=True),
    StructField('name',     StringType(),  nullable=True),
    StructField('amount',   DoubleType(),  nullable=True),
])

df = spark.read.schema(schema).option('header', True).csv('/path/to/file.csv')
```

## write.partitionBy vs repartition

| | `write.partitionBy('col')` | `repartition(n, 'col')` |
|--|---------------------------|-------------------------|
| Effect | Creates directory structure on storage | Reshuffles in-memory partitions |
| Purpose | Storage layout for partition pruning at read | Parallelism and data distribution |
| Output | One folder per distinct key value | `n` in-memory partitions |
| Files per partition | May produce many small files | Controlled by `n` |

```python
# write.partitionBy → /output/year=2024/month=1/part-00000.parquet
df.write.partitionBy('year', 'month').parquet('/output')

# repartition → in-memory shuffle; does NOT change directory structure
df.repartition(4, 'year').write.parquet('/output')
```

## Real-World Scenarios

<details>
<summary>Scenario 1: Production CSV ingest with explicit schema and null handling</summary>

**Situation:**
A partner delivers a daily CSV with a known schema. Some columns contain `'N/A'` for missing values and dates in `MM/dd/yyyy` format. A production pipeline must read this reliably without `inferSchema`.

**Code:**
```python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField('order_id',   IntegerType(), True),
    StructField('customer',   StringType(),  True),
    StructField('amount',     DoubleType(),  True),
    StructField('order_date', StringType(),  True),   # read as string, cast after
    StructField('region',     StringType(),  True),
])

df = spark.read \
    .schema(schema) \
    .option('header', True) \
    .option('nullValue', 'N/A') \
    .option('dateFormat', 'MM/dd/yyyy') \
    .csv('/landing/orders/2024-12-01.csv')

# Cast date column after read
df = df.withColumn('order_date',
    F.to_date(col('order_date'), 'MM/dd/yyyy')
)

df.printSchema()
df.show()
```

**Exam Sub-topic:** Explicit schema vs `inferSchema`; `nullValue` option; date format handling
</details>

<details>
<summary>Scenario 2: Partitioned Parquet write for a time-series reporting table</summary>

**Situation:**
A daily ETL job writes a sales summary to Parquet partitioned by `year` and `month`. Downstream queries always filter on a single month, so partition pruning must be active.

**Code:**
```python
# Add partition columns
summary = daily_sales \
    .withColumn('year',  F.year(col('sale_date'))) \
    .withColumn('month', F.month(col('sale_date')))

# Write partitioned Parquet
summary.write \
    .mode('overwrite') \
    .partitionBy('year', 'month') \
    .parquet('/warehouse/sales_summary')

# Result: /warehouse/sales_summary/year=2024/month=1/part-*.parquet

# Read with partition pruning — only year=2024/month=3/ is scanned
mar_2024 = spark.read \
    .parquet('/warehouse/sales_summary') \
    .filter(col('year') == 2024) \
    .filter(col('month') == 3)

mar_2024.explain()   # shows PartitionFilters in physical plan
```

**Exam Sub-topic:** `partitionBy` on write creates directory hierarchy; partition pruning on read; `PartitionFilters` in `explain()`
</details>

<details>
<summary>Scenario 3: Append mode for streaming-like incremental writes</summary>

**Situation:**
An hourly job appends new events to an existing Delta table. The job must not overwrite historical data.

**Code:**
```python
# Read only the new hour's events
new_events = spark.read.json('/landing/events/2024-12-01/hour=14/')

# Append to the existing Delta table
new_events.write \
    .mode('append') \
    .format('delta') \
    .save('/warehouse/events')

# Verify: total row count grows each hour
print(spark.read.format('delta').load('/warehouse/events').count())
```

**Exam Sub-topic:** `mode('append')` adds rows without deleting existing data; Delta format
</details>

<details>
<summary>Scenario 4: Write modes comparison — overwrite vs ignore for idempotent jobs</summary>

**Situation:**
A job may be re-run if it fails. For most aggregation outputs, `overwrite` is correct. For a staging area that should never be overwritten by a rerun, `ignore` is safer.

**Code:**
```python
# Aggregation output — always recalculate from scratch, safe to overwrite
monthly_summary.write.mode('overwrite').parquet('/reports/monthly_summary')

# Raw backup — once written, never re-written (ignore is a no-op if exists)
raw_snapshot.write.mode('ignore').parquet('/backup/raw/2024-12-01')

# Strict job — fail loudly if output already exists (catches accidental reruns)
output.write.mode('error').parquet('/strict_output/2024-12-01')
# → AnalysisException if path already exists
```

**Exam Sub-topic:** All four write modes — overwrite, append, ignore, error; default is `error`
</details>

<details>
<summary>Scenario 5: Save as managed table and query with SQL</summary>

**Situation:**
A DataFrame result is written as a managed Delta table so it can be queried by SQL analysts and downstream notebooks using `spark.sql()` or `spark.table()`.

**Code:**
```python
# Write and register as a managed table in the default database
enriched_sales.write \
    .mode('overwrite') \
    .format('delta') \
    .saveAsTable('analytics.enriched_sales')

# Read back as a DataFrame
df = spark.table('analytics.enriched_sales')

# Or query with SQL
spark.sql("""
    SELECT region, category,
           SUM(amount) AS total_sales,
           COUNT(*)    AS num_transactions
    FROM   analytics.enriched_sales
    WHERE  year = 2024
    GROUP  BY region, category
    ORDER  BY total_sales DESC
""").show()
```

**Exam Sub-topic:** `saveAsTable()` registers in Hive metastore; `spark.table('name')` reads back; Delta format as default for managed tables
</details>

## Exam Cheat Sheet

| Concept | Key Exam Fact |
|---------|---------------|
| `format('csv')` vs `.csv()` | **Identical result** — different syntax |
| Default write mode | `'error'` / `'errorIfExists'` — fails if path exists |
| `mode('overwrite')` | Deletes existing data and writes fresh |
| `mode('append')` | Adds rows to existing data |
| `mode('ignore')` | Silent no-op if path/table exists |
| CSV `header` default | `False` — column names become `_c0`, `_c1` without it |
| `inferSchema=True` cost | Causes an **extra scan** of the data |
| Explicit schema benefit | No extra scan; exact types; production-safe |
| Parquet schema | **Embedded in file** — no `inferSchema` needed |
| `multiLine=True` (JSON) | Entire file is one JSON object (not one-JSON-per-line) |
| `partitionBy('col')` on write | Creates `/col=val/` directory hierarchy on storage |
| Partition pruning | Reading a partitioned path + filter on partition col → scans only matching dirs |
| `partitionBy` vs `repartition` | `partitionBy` = storage dirs; `repartition` = in-memory partitions |
| `coalesce(1)` before write | Produces a **single output file** |
| `saveAsTable('name')` | Writes **and** registers in Hive metastore |
| `spark.table('name')` | Reads a registered table as a DataFrame |
| Delta read | `spark.read.format('delta').load(path)` |
| `nullValue` option (CSV) | String in file to interpret as `NULL` |
| Bad cast on read | Returns `null` — does NOT raise an error |

---

## Practice Q&A

<details>
<summary>Q1: What is the default write mode and what happens if you write to an existing path?</summary>

**A:** The default write mode is `'error'` (also called `'errorIfExists'`). If the output path or table already exists, Spark raises an `AnalysisException`. You must explicitly choose a different mode:
- `'overwrite'` to replace existing data
- `'append'` to add to existing data
- `'ignore'` to silently skip if already exists
</details>

<details>
<summary>Q2: What is the difference between write.partitionBy('col') and repartition('col')?</summary>

**A:**
- **`write.partitionBy('col')`** creates a **directory hierarchy** on storage: `/output/col=val1/`, `/output/col=val2/`, etc. This enables partition pruning when reading — Spark reads only the matching directories when you filter on that column.
- **`repartition(n, 'col')`** reshuffles data into `n` **in-memory partitions** by hashing on the column. It does not change the output directory structure.

They solve different problems: `partitionBy` is for efficient reads; `repartition` is for parallelism and data distribution.
</details>

<details>
<summary>Q3: Why prefer an explicit StructType schema over inferSchema=True in production?</summary>

**A:**
1. **Performance:** `inferSchema=True` triggers an **extra full scan** of the data to determine types. An explicit schema avoids this.
2. **Correctness:** Schema inference can guess wrong — it may read integers as `LongType`, or fail to parse dates correctly.
3. **Stability:** An explicit schema is self-documenting and won't silently change behaviour if the source data format changes.

In production pipelines, always define the schema explicitly.
</details>

<details>
<summary>Q4: You read a Parquet file written from a DataFrame. Do you need inferSchema=True?</summary>

**A:** **No.** Parquet is a **self-describing format** — the schema is embedded in the file metadata. Spark reads it automatically without an extra scan. `inferSchema` is only relevant for text-based formats like CSV.
</details>

<details>
<summary>Q5: What is the effect of calling coalesce(1) before writing?</summary>

**A:** `coalesce(1)` reduces the DataFrame to a single in-memory partition. When written to disk, this produces **one output file** (plus metadata files). This is useful when a downstream system expects a single file.

Important trade-offs:
- `coalesce(1)` tries to avoid a full shuffle (it merges partitions on the same executor when possible).
- The single file cannot be read in parallel — bad for large DataFrames.
- For large data, prefer `repartition(n)` to balance write parallelism, or keep default partitions.
</details>